In [9]:
# TÁCH X VÀ y
# X = dữ liệu đầu vào
# y = nhãn cần dự đoán (Loan_Status)

X = df.drop("Loan_Status", axis=1).values
y = df["Loan_Status"].values

print("Shape X:", X.shape)
print("Shape y:", y.shape)

# HÀM CHUẨN HÓA DỮ LIỆU THỦ CÔNG

def standardize_data(X_train, X_test):

    mean = np.mean(X_train, axis=0)

    std = np.std(X_train, axis=0)

    std[std == 0] = 1

    X_train_scaled = (X_train - mean) / std

    X_test_scaled = (X_test - mean) / std

    return X_train_scaled, X_test_scaled

#  CHIA TRAIN / TEST THỦ CÔNG
# 80% train - 20% test

np.random.seed(42)

indices = np.random.permutation(len(X))

split_index = int(len(X) * 0.8)

train_idx = indices[:split_index]
test_idx = indices[split_index:]

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

X_train, X_test = standardize_data(X_train, X_test)

print("\nSố mẫu train:", len(X_train))
print("Số mẫu test:", len(X_test))
np.save('train_idx.npy', train_idx)
np.save('test_idx.npy', test_idx)

# HÀM TÍNH KHOẢNG CÁCH EUCLID
# Công thức:
# d = sqrt((x2-x1)^2 + (y2-y1)^2 + ...)

def euclidean_distance(x1, x2):

    distance = 0

    for i in range(len(x1)):
        distance += (x1[i] - x2[i]) ** 2

    return distance ** 0.5


# HÀM DỰ ĐOÁN 1 DÒNG DỮ LIỆU

def predict_one(X_train, y_train, x_test, k=5):

    distances = []

    # Tính khoảng cách từ điểm test tới toàn bộ train
    for i in range(len(X_train)):

        d = euclidean_distance(X_train[i], x_test)

        distances.append((d, y_train[i]))

    # Sắp xếp theo khoảng cách tăng dần
    distances.sort(key=lambda x: x[0])

    # Lấy K điểm gần nhất
    k_nearest = distances[:k]

    # Lấy nhãn của K điểm
    labels = [label for distance, label in k_nearest]

    # Majority vote (bỏ phiếu đa số)
    prediction = Counter(labels).most_common(1)[0][0]

    return prediction


# HÀM DỰ ĐOÁN TOÀN BỘ TEST SET

def predict(X_train, y_train, X_test, k=5):

    predictions = []

    for x in X_test:

        pred = predict_one(X_train, y_train, x, k)

        predictions.append(pred)

    return np.array(predictions)


# HÀM TÍNH ACCURACY
# Accuracy = số dự đoán đúng / tổng số mẫu

def accuracy_score(y_true, y_pred):

    correct = 0

    for i in range(len(y_true)):

        if y_true[i] == y_pred[i]:
            correct += 1

    return correct / len(y_true)


# CHẠY MÔ HÌNH KNN

k = 3

y_pred = predict(X_train, y_train, X_test, k)


# TÍNH ĐỘ CHÍNH XÁC

accuracy = accuracy_score(y_test, y_pred)

print("\n========== KẾT QUẢ MÔ HÌNH ==========")
print("Giá trị K =", k)
print("Accuracy =", round(accuracy * 100, 2), "%")


# HIỂN THỊ MỘT SỐ KẾT QUẢ DỰ ĐOÁN

print("\n10 kết quả dự đoán đầu tiên:")

for i in range(10):

    du_doan = "Được duyệt" if y_pred[i] == 1 else "Từ chối"

    thuc_te = "Được duyệt" if y_test[i] == 1 else "Từ chối"

    print(
        "Dự đoán:", du_doan,
        "| Thực tế:", thuc_te
    )

# TÍNH F1-SCORE THỦ CÔNG

tp = 0
fp = 0
fn = 0

for i in range(len(y_test)):

    if y_pred[i] == 1 and y_test[i] == 1:
        tp += 1

    elif y_pred[i] == 1 and y_test[i] == 0:
        fp += 1

    elif y_pred[i] == 0 and y_test[i] == 1:
        fn += 1


# PRECISION

precision = tp / (tp + fp)


# RECALL

recall = tp / (tp + fn)


# F1-SCORE

f1_score = 2 * (precision * recall) / (precision + recall)

print("\nF1-score =", round(f1_score, 4))


# LƯU KẾT QUẢ CHO PHẦN 4

pd.DataFrame({
    'k': [k],
    'accuracy': [accuracy],
    'f1_score': [f1_score]
}).to_csv('ket_qua_phan3.csv', index=False)

print('✅ Đã lưu ket_qua_phan3.csv cho Phần 4 ')

Shape X: (571, 6)
Shape y: (571,)

Số mẫu train: 456
Số mẫu test: 115

========== KẾT QUẢ MÔ HÌNH ==========
Giá trị K = 3
Accuracy = 75.65 %

10 kết quả dự đoán đầu tiên:
Dự đoán: Được duyệt | Thực tế: Được duyệt
Dự đoán: Từ chối | Thực tế: Từ chối
Dự đoán: Từ chối | Thực tế: Từ chối
Dự đoán: Được duyệt | Thực tế: Từ chối
Dự đoán: Được duyệt | Thực tế: Từ chối
Dự đoán: Được duyệt | Thực tế: Được duyệt
Dự đoán: Được duyệt | Thực tế: Từ chối
Dự đoán: Được duyệt | Thực tế: Được duyệt
Dự đoán: Từ chối | Thực tế: Từ chối
Dự đoán: Được duyệt | Thực tế: Được duyệt

F1-score = 0.8333
✅ Đã lưu ket_qua_phan3.csv cho Phần 4 


In [5]:
import pandas as pd
import numpy as np
from collections import Counter

df = pd.read_csv("clean_no_outlier.csv")